In [ ]:
# Start program timer
import time
start_time = time.time()

In [ ]:
# Import required modules
from transcript_configurator import TranscriptConfigurator # Contains 2 classes needed to convert transcripts into pandas dataframes
from requester import GPTConcurrentRequester # Uses up to 25 workers concurrently to iterate through transcript and send GPT request for EPL info
from calculator import ScoreboardCalculator # Calculates transcript attributes and defines scoring functions
# from scraper import TranscriptScraper # Extraxts transcripts and metadata and writes them to labled text files named by transcript date, enclosed in directories by transcript year
# [to-do: Build class to preprocess data for optimal processing]
from debate_dict import debate_dict # Predefined transcript metadata; includes URLs used by TranscriptParser
from dotenv import load_dotenv # Loads environment variables
from utils import pickle
import swifter
import pandas as pd # Used for dataframe exploration
import dtale # Opens a robust interactive dataframe GUI
import spacy # Used for tokenization
import nltk # Used for named entitity recognition (NER)
import os # Loads environment variables

In [ ]:
# Extracts all available debate transcripts from debates.orgs and preprocesses a single transcript from 2008-9-26
# TranscriptScraper().scrape_all_content_and_save(debate_dict)

# to-do: Build Class to clean all scraped content

# to-do: Build Class to use scraped metadata to create dictionaries of known speakers by transcript

# to-do: Build Class to transfer post-processed data to database

In [ ]:
# Sets environment variables and initializes critical objects
load_dotenv('.env')
open_ai_keys = [os.getenv(f"OPENAI_KEY_{i}") for i in range(1, 26)]
nlp = spacy.load("en_core_web_lg")
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

In [ ]:
# Loads transcripts and speaker names
original_transcript_path = "../clean/2008-09-26-debate-preclean_clean_.txt"
known_speakers = {'LEHRER', 'OBAMA', 'MCCAIN'}

In [ ]:
# Configures transcript scoreboard
configurator = TranscriptConfigurator()
empty_scoreboard = configurator.get_updated_scoreboard(original_transcript_path, known_speakers)

In [ ]:
# Initalizes base scores
scoreboard_calculator = ScoreboardCalculator(empty_scoreboard, nlp, english_vocab)

In [ ]:
# Requests GPT-4 for EPL information
gpt_requester = GPTConcurrentRequester(empty_scoreboard, open_ai_keys, original_transcript_path, scoreboard_calculator)
final_scoreboard = gpt_requester.get_epl()

In [ ]:
# Updates scoreboard values with EPL info
updated_final_scoreboard = scoreboard_calculator.update_scoreboard()

In [ ]:
n_base_score = len(updated_final_scoreboard)
mean_base_score = updated_final_scoreboard['base_score'].mean()
std_base_score = updated_final_scoreboard['base_score'].std()
updated_final_scoreboard['base_score_z_score'] = (updated_final_scoreboard['base_score'] - mean_base_score)/std_base_score


In [ ]:
updated_final_scoreboard.to_pickle('updated_final_scoreboard.pkl')

In [ ]:
# Displays final scoreboard dataframe using dtale
x = dtale.show(updated_final_scoreboard)
x.open_browser()

In [ ]:
# Prints total execution time
elapsed = (time.time() - start_time)
print('Execution time:', time.strftime("%H:%M:%S", time.gmtime(elapsed)))